In [ ]:
# Permite importar src/ desde la carpeta notebooks/
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import DIR_DATA_RAW

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [ ]:
from src.config import DIR_DATA_PROC
from src.variables import construir_variables, verificar_sin_futuro, FEATURES
from src.etiquetas import etiquetar

tel_v = pd.read_parquet(DIR_DATA_PROC / "telemetria_viajes.parquet")
vf = pd.read_parquet(DIR_DATA_PROC / "ventanas_calas.parquet")

# 1. Variables y prueba de fuga con datos reales
tel_f = construir_variables(tel_v)
print("Sin datos del futuro:", verificar_sin_futuro(tel_v.sample(5000, random_state=1).sort_values(["ep", "fecha"]), n=50))
display(tel_f[FEATURES].isna().mean().round(3).sort_values(ascending=False).head(8).to_frame("% vacíos"))

# 2. Etiquetas con transiciones
tel_e = etiquetar(tel_f, vf)
display(tel_e["Etiqueta"].value_counts().to_frame("registros"))
display(tel_e.groupby("Etiqueta")[["Velocidad", "Velocidad_delta", "rpm", "rpm_delta",
                                   "gal_h", "gal_h_delta", "GiroAcum3"]].median().round(1))

tel_e.to_parquet(DIR_DATA_PROC / "telemetria_etiquetada.parquet", index=False)

In [ ]:
e = tel_e.copy()
e["sig_etq"] = e.groupby("Tramo")["Etiqueta"].shift(-1)
e["pos_entrada"] = np.where(e["Etiqueta"] != "entrada_cala", None,
                     np.where(e["sig_etq"] == "cala", "justo antes de la cala", "2 registros antes"))
cols6a = ["Velocidad", "Velocidad_lag1", "Velocidad_delta",
          "rpm", "rpm_lag1", "rpm_delta", "gal_h", "gal_h_delta", "GiroAcum3"]
display(e[e["Etiqueta"] == "entrada_cala"].groupby("pos_entrada")[cols6a].median().round(1))

In [ ]:
from src.paradas import detectar_paradas, marcar_calas_registradas
viajes = pd.read_parquet(DIR_DATA_PROC / "viajes.parquet")

par = detectar_paradas(tel_f)
par = marcar_calas_registradas(par, vf, viajes)
par["Clase"] = np.select(
    [par["Tipo"] == "cala_registrada", par["dist_puerto_km"] <= 5],
    ["positiva (cala registrada)", "negativa (puerto)"],
    "sin etiqueta")

display(par["Clase"].value_counts())
cols = ["duracion_min", "galh_mediana", "rpm_mediana", "rpm_std", "vel_antes", "vel_despues",
        "giro_antes", "giro_despues", "deriva_km", "hora_inicio"]
display(par.groupby("Clase")[cols].median().T.round(1))

fig, axes = plt.subplots(4, 3, figsize=(16, 14))
for ax, col in zip(axes.flat, cols):
    lim = par[col].quantile(0.98)
    sns.histplot(data=par[par[col] <= lim], x=col, hue="Clase", stat="density",
                 common_norm=False, bins=30, element="step", ax=ax)
for ax in axes.flat[len(cols):]:
    ax.axis("off")
plt.tight_layout()

In [ ]:
from src.modelo_paradas import entrenar_pu, clasificar, VARIABLES
from sklearn.inspection import permutation_importance

p, c, modelo = entrenar_pu(par)
p["Prediccion"] = clasificar(p["P_cala"])
print(f"c = P(una cala se registre) = {c:.2f}")

# 1. ¿Reconoce las calas registradas que no vio?
pos = p[p["s"] == 1]
print(f"Calas registradas reconocidas (P ≥ 0.5): {(pos['P_cala'] >= 0.5).mean():.1%}")

# 2. Resultado por clase de parada
display(pd.crosstab(p["Clase"], p["Prediccion"], margins=True))

# 3. Distribución de la probabilidad en las sin etiqueta: ¿se separa en dos grupos?
p.loc[p["Clase"] == "sin etiqueta", "P_cala"].hist(bins=40, color="#0065AC")
plt.title("P(cala) de paradas sin etiqueta"); plt.xlabel("P(cala)")

# 4. Coherencia por viaje: calas nuevas por viaje (no debería dispararse)
nuevas = p[(p["s"] == 0) & (p["Prediccion"] == "cala probable") & p["IdViaje"].notna()]
print("Calas nuevas por viaje:", nuevas.groupby("IdViaje").size().describe().round(1).to_dict())
print("Calas probables fuera de viaje (mar sin faena):",
      ((p["s"] == 0) & (p["Prediccion"] == "cala probable") & p["IdViaje"].isna()).sum())

# 5. ¿En qué se fija el modelo?
X = p[VARIABLES]
imp = permutation_importance(modelo, X, p["s"], n_repeats=5, random_state=0, scoring="roc_auc")
display(pd.Series(imp.importances_mean, index=VARIABLES).sort_values(ascending=False).round(3).to_frame("importancia"))

p.to_parquet(DIR_DATA_PROC / "paradas_pu.parquet", index=False)
p[p["Prediccion"] == "dudosa"].to_csv(DIR_DATA_PROC / "paradas_por_revisar.csv", index=False)

In [ ]:
from src.evaluacion_pu import evaluar
from src.modelo_paradas import entrenar_pu, clasificar, VARIABLES

# 1. Indicadores y punto de corte del modelo actual
tabla, cortes, mejor = evaluar(p, c)
display(tabla)
print("Puntos de corte óptimos:", {k: float(v) for k, v in mejor.items()})
plt.show()

u = mejor["Lee-Liu (PU, recomendado)"]
display(cortes.set_index("umbral").loc[[0.3, 0.5, 0.7, u]].round(3))

# 2. Prueba sin deriva_km (¿el modelo depende de la regla del puerto?)
import src.modelo_paradas as mp
mp.VARIABLES = [v for v in VARIABLES if v != "deriva_km"]
p2, c2, modelo2 = mp.entrenar_pu(par)
tabla2, _, mejor2 = evaluar(p2, c2)
mp.VARIABLES = VARIABLES                                   # restaurar

display(pd.concat({"con deriva": tabla, "sin deriva": tabla2}, axis=1))
print("Corte sin deriva:", float(mejor2["Lee-Liu (PU, recomendado)"]))

In [ ]:
import joblib, datetime
from src.config import DIR_MODELOS
from src.modelo_paradas import entrenar_pu, clasificar, VARIABLES

p, c, modelo = entrenar_pu(par)
p["Prediccion"] = clasificar(p["P_cala"])
display(pd.crosstab(p["Clase"], p["Prediccion"], margins=True))

# El modelo y TODO lo necesario para usarlo en producción, en un solo archivo
DIR_MODELOS.mkdir(exist_ok=True)
joblib.dump({
    "modelo": modelo, "c": c, "variables": VARIABLES,
    "umbral_cala": 0.5, "umbral_revision": 0.2,
    "version": "paradas_pu_v1", "entrenado": datetime.datetime.now().isoformat(timespec="minutes"),
    "metricas": {"auc_pu": 0.88, "auc_vs_puerto": 0.99, "recall_umbral_0.2": 0.95},
}, DIR_MODELOS / "paradas_pu_v1.joblib")

p.to_parquet(DIR_DATA_PROC / "paradas_pu.parquet", index=False)
p[p["Prediccion"] == "posible cala"].to_csv(DIR_DATA_PROC / "paradas_por_revisar.csv", index=False)
print("Calas nuevas (no registradas):", ((p["s"] == 0) & (p["Prediccion"] == "cala")).sum())